# Firma Digital RSA — Demo educativa

Este notebook es una versión simple para explicar y grabar el video.

La idea principal:

1. Generamos una **llave pública** y una **llave privada**.
2. Convertimos el mensaje a un **hash SHA-256**.
3. Firmamos el hash con la **llave privada**.
4. Verificamos la firma con la **llave pública**.
5. Probamos qué pasa si el mensaje cambia o si se usa una llave incorrecta.

> Importante: esto es una implementación educativa. No se debe usar en producción.

## 1. Importamos librerías permitidas

Usamos solo librerías generales de Python:

- `random`: para generar números aleatorios.
- `hashlib`: para calcular SHA-256.
- `math`: para apoyo matemático básico.

No usamos librerías criptográficas como `rsa`, `cryptography` o `PyCryptodome`.

In [ ]:
import random
import hashlib
import math

## 2. Funciones matemáticas básicas

RSA necesita dos operaciones importantes:

- Calcular el **máximo común divisor**.
- Calcular el **inverso modular**.

No necesitas memorizar toda la matemática. Para defenderlo, basta con saber esto:

- `mcd(e, phi)` revisa que `e` sea compatible con `phi`.
- `inverso_modular(e, phi)` calcula `d`, que será parte de la llave privada.

In [ ]:
def mcd(a, b):
    """Calcula el máximo común divisor usando el algoritmo de Euclides."""
    while b != 0:
        a, b = b, a % b
    return a


def inverso_modular(e, phi):
    """
    Calcula d tal que:
        e * d ≡ 1 mod phi

    En RSA, ese valor d se usa como exponente privado.
    """
    phi_original = phi
    x0, x1 = 0, 1

    while e > 1:
        cociente = e // phi
        e, phi = phi, e % phi
        x0, x1 = x1 - cociente * x0, x0

    if x1 < 0:
        x1 += phi_original

    return x1

## 3. Generación de números primos

RSA necesita dos números primos grandes:

- `p`
- `q`

Con ellos se calcula:

```text
n = p * q
```

En este notebook usamos una prueba de primalidad sencilla basada en Miller-Rabin.

In [ ]:
def es_primo(n, rondas=10):
    """Prueba probabilística Miller-Rabin para revisar si n probablemente es primo."""
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False

    # Escribimos n-1 como 2^r * d
    r = 0
    d = n - 1

    while d % 2 == 0:
        r += 1
        d //= 2

    for _ in range(rondas):
        a = random.randrange(2, n - 1)
        x = pow(a, d, n)

        if x == 1 or x == n - 1:
            continue

        for _ in range(r - 1):
            x = pow(x, 2, n)

            if x == n - 1:
                break
        else:
            return False

    return True


def primo_aleatorio(bits):
    """Genera un número primo aleatorio con la cantidad de bits indicada."""
    while True:
        candidato = random.getrandbits(bits)

        # Forzamos que sea impar y que tenga el tamaño correcto en bits.
        candidato = candidato | (1 << (bits - 1)) | 1

        if es_primo(candidato):
            return candidato

## 4. Generación de llaves RSA

La función `generar_llaves()` produce:

```text
Llave pública  = (n, e)
Llave privada  = (n, d)
```

La **llave pública** se puede compartir y sirve para verificar.

La **llave privada** debe guardarse en secreto y sirve para firmar.

In [ ]:
def generar_llaves(bits=512):
    print("Generando llaves RSA...")

    # 1. Generar dos primos secretos
    p = primo_aleatorio(bits // 2)
    q = primo_aleatorio(bits // 2)

    while p == q:
        q = primo_aleatorio(bits // 2)

    # 2. Calcular n
    n = p * q

    # 3. Calcular phi(n)
    phi = (p - 1) * (q - 1)

    # 4. Elegir e
    e = 65537

    if mcd(e, phi) != 1:
        e = 3
        while mcd(e, phi) != 1:
            e += 2

    # 5. Calcular d
    d = inverso_modular(e, phi)

    llave_publica = (n, e)
    llave_privada = (n, d)

    print("\nValores educativos generados:")
    print(f"p = {p}")
    print(f"q = {q}")
    print(f"n = p*q = {n}")
    print(f"phi(n) = {phi}")
    print(f"e = {e}")
    print(f"d = {d}")

    print("\nLlave pública  (n, e):", llave_publica)
    print("Llave privada  (n, d):", llave_privada)

    print("\nExplicación:")
    print("- La llave pública se puede compartir y se usa para verificar firmas.")
    print("- La llave privada debe mantenerse secreta y se usa para firmar.")

    return llave_publica, llave_privada

## 5. Preparación del mensaje

No firmamos el texto directamente.

Primero calculamos:

```text
SHA-256(mensaje)
```

Ese hash funciona como una huella digital del mensaje.

Si el mensaje cambia aunque sea un carácter, el hash cambia y la firma deja de ser válida.

In [ ]:
def hash_mensaje(mensaje):
    """
    Convierte un mensaje de texto a SHA-256.
    Regresa:
    - hash en hexadecimal, para mostrarlo.
    - hash como entero, para usarlo en RSA.
    """
    digest = hashlib.sha256(mensaje.encode("utf-8")).digest()
    hash_hex = digest.hex()
    hash_entero = int.from_bytes(digest, "big")

    return hash_hex, hash_entero


def validar_mensaje(mensaje):
    """Validación simple para evitar mensajes vacíos o datos incorrectos."""
    if not isinstance(mensaje, str):
        return False, "El mensaje debe ser texto tipo str."

    if mensaje.strip() == "":
        return False, "El mensaje no puede estar vacío."

    return True, "OK"

## 6. Firma digital

Esta es la parte más importante para la defensa.

La firma se calcula así:

```text
firma = hash^d mod n
```

Donde:

- `hash` es el SHA-256 del mensaje.
- `d` viene de la llave privada.
- `n` es parte de ambas llaves.

En código, la línea central es:

```python
firma = pow(h, d, n)
```

In [ ]:
def firmar(mensaje, llave_privada):
    ok, razon = validar_mensaje(mensaje)

    if not ok:
        raise ValueError(razon)

    n, d = llave_privada

    hash_hex, h = hash_mensaje(mensaje)

    if h >= n:
        raise ValueError("El hash es mayor o igual que n. Se necesitan llaves más grandes.")

    # Lógica principal de firma digital RSA
    firma = pow(h, d, n)

    print("Mensaje:", mensaje)
    print("SHA-256:", hash_hex)
    print("Hash como entero:", h)
    print("\nFirma generada:", firma)

    return firma

## 7. Verificación de firma

Para verificar, usamos la llave pública.

La verificación calcula:

```text
H' = firma^e mod n
```

Luego vuelve a calcular el hash del mensaje recibido.

Si ambos valores son iguales, la firma es válida.

La línea central de verificación es:

```python
h_recuperado = pow(firma, e, n)
```

In [ ]:
def verificar(mensaje, firma, llave_publica):
    ok, razon = validar_mensaje(mensaje)

    if not ok:
        print("Mensaje inválido:", razon)
        return False

    if not isinstance(firma, int) or firma < 0:
        print("Firma inválida: debe ser un entero positivo.")
        return False

    n, e = llave_publica

    if firma >= n:
        print("Firma inválida: la firma es mayor o igual que n.")
        return False

    # Lógica principal de verificación RSA
    h_recuperado = pow(firma, e, n)

    hash_hex, h_mensaje = hash_mensaje(mensaje)

    print("Mensaje recibido:", mensaje)
    print("SHA-256 del mensaje recibido:", hash_hex)
    print("Hash del mensaje:", h_mensaje)
    print("Hash recuperado desde la firma:", h_recuperado)

    if h_recuperado == h_mensaje:
        print("\nRESULTADO: Firma válida.")
        return True
    else:
        print("\nRESULTADO: Firma inválida.")
        return False

## 8. Demo principal

Esta es la parte que puedes usar para grabar el video.

Primero generamos llaves. Luego firmamos un mensaje y lo verificamos.

In [ ]:
llave_publica, llave_privada = generar_llaves(bits=512)

In [ ]:
mensaje = "Transferencia autorizada: $500 a cuenta 4321"

firma = firmar(mensaje, llave_privada)

In [ ]:
verificar(mensaje, firma, llave_publica)

## 9. Caso: mensaje alterado

Aquí usamos la firma original, pero cambiamos el mensaje.

Esto debe fallar porque el hash del mensaje alterado ya no coincide con el hash recuperado desde la firma.

In [ ]:
mensaje_alterado = "Transferencia autorizada: $900 a cuenta 4321"

verificar(mensaje_alterado, firma, llave_publica)

## 10. Caso: llave pública incorrecta

Aquí generamos otro par de llaves y tratamos de verificar con la llave pública equivocada.

Esto debe fallar porque la firma fue creada con otra llave privada.

In [ ]:
llave_publica_falsa, llave_privada_falsa = generar_llaves(bits=512)

verificar(mensaje, firma, llave_publica_falsa)

## 11. Caso: mensaje vacío

El sistema debe rechazar mensajes vacíos.

In [ ]:
try:
    firmar("", llave_privada)
except ValueError as error:
    print("Error capturado correctamente:", error)

verificar("", firma, llave_publica)

## 12. Caso: firma mal formada

Aquí intentamos verificar usando una firma inválida.

In [ ]:
firma_mala = "esto no es una firma"

verificar(mensaje, firma_mala, llave_publica)

## 13. Respuestas rápidas para defensa

Puedes usar estas ideas en el video o en clase:

**¿Qué parte del código firma realmente?**

```python
firma = pow(h, d, n)
```

**¿Qué parte verifica realmente?**

```python
h_recuperado = pow(firma, e, n)
```

**¿Qué pasa si el mensaje cambia?**

Cambia el hash SHA-256, entonces la firma ya no coincide y la verificación falla.

**¿Qué pasa si uso otra llave pública?**

La verificación falla porque esa llave pública no corresponde a la llave privada que creó la firma.

**¿Cuál es la principal limitación?**

Es una implementación educativa. Usa RSA sin padding seguro, llaves pequeñas y `random`, por eso no debe usarse en producción.